# Benchmarking XLinear vs. NHITS on ETTm1

**Module 5 | Modern Time Series Forecasting with NeuralForecast**

**Estimated time:** 12–15 minutes (including training both models)

---

## What You Will Do

1. Train XLinear and NHITS on identical ETTm1 train/val/test splits
2. Use `.cross_validation()` for a fair, reproducible comparison
3. Report MAE and MSE for both models across all 7 series
4. Plot side-by-side forecast comparisons for 3 representative series
5. Identify when to choose XLinear vs. NHITS

---

## Why Fair Comparison Matters

A valid model comparison requires:
- **Same train/val/test splits** — both models see identical training data
- **Same evaluation metric** — MAE and MSE computed on the same test windows
- **Same forecast horizon** — h=96 for both models
- **Out-of-sample forecasts** — never evaluate on training data

NeuralForecast's `.cross_validation()` guarantees all of these by construction.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from neuralforecast import NeuralForecast
from neuralforecast.models import XLinear, NHITS
from datasetsforecast.long_horizon import LongHorizon
from utilsforecast.losses import mae, mse
from utilsforecast.evaluation import evaluate

print("Libraries loaded.")

## 2. Load ETTm1 and Set Benchmark Parameters

In [ ]:
Y_df, _, _ = LongHorizon.load(directory="./data", group="ETTm1")
Y_df["ds"] = pd.to_datetime(Y_df["ds"])

# Benchmark parameters — both models use these identically
H = 96           # 24 hours at 15-min resolution
FREQ = "15min"
VAL_SIZE = 11520  # ~120 days
TEST_SIZE = 11520  # ~120 days

print(f"ETTm1: {Y_df['unique_id'].nunique()} series, {Y_df['ds'].nunique():,} timestamps")
print(f"Forecast horizon: {H} steps ({H * 15 // 60} hours)")
print(f"Val/test size: {VAL_SIZE:,} steps each")

## 3. Define Both Models

**NHITS** is a univariate hierarchical model — it trains one model per series independently. It has no cross-variable learning.

**XLinear** is a multivariate model — it trains one model for all 7 series simultaneously, with cross-variable associations learned via VGM.

In [ ]:
# NHITS: univariate, hierarchical MLP
# - input_size=192 gives longer context (2x horizon) — typical NHITS setting
# - n_freq_downsample defines the hierarchical interpolation stacks
nhits_model = NHITS(
    h=H,
    input_size=192,
    n_freq_downsample=[2, 1, 1],
    dropout_prob_theta=0.0,
    learning_rate=1e-3,
    batch_size=32,
    max_steps=1000,
)

# XLinear: multivariate, gated MLP
# - n_series=7 enables VGM cross-variable learning
# - All 7 ETTm1 series trained jointly
xlinear_model = XLinear(
    h=H,
    input_size=H,          # input_size = h is the standard benchmark setting
    n_series=7,
    hidden_size=512,
    temporal_ff=256,
    channel_ff=21,
    head_dropout=0.5,
    embed_dropout=0.2,
    learning_rate=1e-4,
    batch_size=32,
    max_steps=2000,
)

print("Models defined.")
print(f"  NHITS:   univariate, max_steps=1000")
print(f"  XLinear: multivariate (n_series=7), max_steps=2000")

## 4. Cross-Validation — Both Models

We train each model separately to isolate their behavior. NeuralForecast supports training multiple models in one call, but separate calls make the comparison cleaner for inspection.

**Expected runtime:**
- NHITS: 2–4 minutes (1000 steps, univariate)
- XLinear: 3–6 minutes (2000 steps, multivariate)

In [ ]:
# Train NHITS
print("Training NHITS...")
nf_nhits = NeuralForecast(models=[nhits_model], freq=FREQ)
cv_nhits = nf_nhits.cross_validation(
    df=Y_df, val_size=VAL_SIZE, test_size=TEST_SIZE
)
print(f"NHITS done. Output shape: {cv_nhits.shape}")

In [ ]:
# Train XLinear
print("Training XLinear...")
nf_xlinear = NeuralForecast(models=[xlinear_model], freq=FREQ)
cv_xlinear = nf_xlinear.cross_validation(
    df=Y_df, val_size=VAL_SIZE, test_size=TEST_SIZE
)
print(f"XLinear done. Output shape: {cv_xlinear.shape}")

## 5. Evaluate Both Models

Filter to the test set (final cutoff) and compute MAE + MSE using `utilsforecast.evaluation.evaluate()`.

In [ ]:
# Filter both cv outputs to test set
test_nhits = cv_nhits[cv_nhits["cutoff"] == cv_nhits["cutoff"].max()].copy().reset_index(drop=True)
test_xlinear = cv_xlinear[cv_xlinear["cutoff"] == cv_xlinear["cutoff"].max()].copy().reset_index(drop=True)

print(f"NHITS test rows: {len(test_nhits):,}")
print(f"XLinear test rows: {len(test_xlinear):,}")

In [ ]:
# Compute metrics
eval_nhits = evaluate(df=test_nhits, metrics=[mae, mse], models=["NHITS"])
eval_xlinear = evaluate(df=test_xlinear, metrics=[mae, mse], models=["XLinear"])

# Summary
nhits_mae = eval_nhits[eval_nhits["metric"] == "mae"]["NHITS"].mean()
nhits_mse = eval_nhits[eval_nhits["metric"] == "mse"]["NHITS"].mean()
xlinear_mae = eval_xlinear[eval_xlinear["metric"] == "mae"]["XLinear"].mean()
xlinear_mse = eval_xlinear[eval_xlinear["metric"] == "mse"]["XLinear"].mean()

print("="*55)
print(f"{'Model':<12} {'MAE':>10} {'MSE':>10}")
print("-"*55)
print(f"{'NHITS':<12} {nhits_mae:>10.4f} {nhits_mse:>10.4f}")
print(f"{'XLinear':<12} {xlinear_mae:>10.4f} {xlinear_mse:>10.4f}")
print("-"*55)
print(f"{'Improvement':<12} {(nhits_mae - xlinear_mae) / nhits_mae * 100:>9.1f}% {(nhits_mse - xlinear_mse) / nhits_mse * 100:>9.1f}%")
print("="*55)
print("(Positive improvement = XLinear wins)")

### 5a. Per-Series Breakdown

The per-series breakdown reveals *where* XLinear's multivariate learning helps most. The OT (oil temperature) series benefits most because its future values are physically caused by current load values.

In [ ]:
# Per-series MAE for both models
mae_nhits_series = eval_nhits[eval_nhits["metric"] == "mae"].set_index("unique_id")["NHITS"]
mae_xlinear_series = eval_xlinear[eval_xlinear["metric"] == "mae"].set_index("unique_id")["XLinear"]

comparison = pd.DataFrame({
    "NHITS MAE": mae_nhits_series,
    "XLinear MAE": mae_xlinear_series,
})
comparison["Improvement %"] = (comparison["NHITS MAE"] - comparison["XLinear MAE"]) / comparison["NHITS MAE"] * 100
comparison["Winner"] = comparison["Improvement %"].apply(lambda x: "XLinear" if x > 0 else "NHITS")
comparison = comparison.sort_values("Improvement %", ascending=False)

print("Per-series MAE comparison (sorted by XLinear improvement):")
print(comparison.to_string(float_format="{:.4f}".format))

### 5b. Visualize the Per-Series Comparison

In [ ]:
SERIES_ORDER = comparison.index.tolist()
x = np.arange(len(SERIES_ORDER))
width = 0.35

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Left: MAE bars
bars_nhits = ax1.bar(x - width/2, comparison["NHITS MAE"], width,
                     label="NHITS", color="#1f77b4", alpha=0.85, edgecolor="white")
bars_xlinear = ax1.bar(x + width/2, comparison["XLinear MAE"], width,
                       label="XLinear", color="#d62728", alpha=0.85, edgecolor="white")

ax1.set_xticks(x)
ax1.set_xticklabels(SERIES_ORDER, fontsize=10)
ax1.set_ylabel("MAE", fontsize=11)
ax1.set_title("MAE per Series: NHITS vs. XLinear", fontsize=12, fontweight="bold")
ax1.legend(fontsize=10)
ax1.grid(axis="y", alpha=0.3)

# Right: Improvement %
bar_colors = ["#2ca02c" if v > 0 else "#ff7f0e" for v in comparison["Improvement %"]]
ax2.bar(SERIES_ORDER, comparison["Improvement %"], color=bar_colors, edgecolor="white", alpha=0.85)
ax2.axhline(0, color="black", linewidth=0.8)
ax2.set_ylabel("XLinear improvement over NHITS (%)", fontsize=10)
ax2.set_title("XLinear Improvement by Series", fontsize=12, fontweight="bold")
ax2.grid(axis="y", alpha=0.3)

for ax in [ax1, ax2]:
    ax.tick_params(axis="x", labelsize=9)

plt.tight_layout()
plt.savefig("benchmark_mae_comparison.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: benchmark_mae_comparison.png")

## 6. Side-by-Side Forecast Plots

Visual comparison of NHITS and XLinear forecasts for 3 series. The key question: does XLinear track the actual values more closely than NHITS, particularly for OT?

In [ ]:
def plot_side_by_side(test_nhits, test_xlinear, series_id, n_days=3):
    """Side-by-side actual vs. forecast for one series."""
    nhits_s = test_nhits[test_nhits["unique_id"] == series_id].copy()
    xlinear_s = test_xlinear[test_xlinear["unique_id"] == series_id].copy()

    cutoff_ts = nhits_s["ds"].max() - pd.Timedelta(days=n_days)
    nhits_s = nhits_s[nhits_s["ds"] >= cutoff_ts]
    xlinear_s = xlinear_s[xlinear_s["ds"] >= cutoff_ts]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 3.5), sharey=True)
    fig.suptitle(f"{series_id} — NHITS vs. XLinear Forecasts (Last {n_days} Days)",
                 fontsize=12, fontweight="bold")

    for ax, df, model_name, color in [
        (ax1, nhits_s, "NHITS", "#1f77b4"),
        (ax2, xlinear_s, "XLinear", "#d62728"),
    ]:
        ax.plot(df["ds"], df["y"], color="black", linewidth=1.2,
                label="Actual", alpha=0.85)
        ax.plot(df["ds"], df[model_name], color=color, linewidth=1.2,
                linestyle="--", label=f"{model_name} forecast", alpha=0.9)
        ax.fill_between(df["ds"], df["y"], df[model_name],
                        alpha=0.15, color=color)
        ax.set_title(model_name, fontsize=11)
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
        ax.tick_params(axis="x", labelsize=8, rotation=20)

    plt.tight_layout()
    return fig


# Plot 3 representative series
for series_id in ["HUFL", "MUFL", "OT"]:
    fig = plot_side_by_side(test_nhits, test_xlinear, series_id, n_days=3)
    fig.savefig(f"forecast_comparison_{series_id}.png", dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Saved: forecast_comparison_{series_id}.png")

## 7. Decision Framework: XLinear vs. NHITS

The benchmark numbers tell you which model wins on ETTm1. But the more useful question is: **when should you choose each model on a new dataset?**

In [ ]:
decision_table = pd.DataFrame({
    "Factor": [
        "Number of series",
        "Cross-series correlation",
        "Physical/causal relationships",
        "Dataset size (training)",
        "Training compute budget",
        "Exogenous features needed",
        "Primary goal",
    ],
    "Choose XLinear when...": [
        "Multiple correlated series (N >= 3)",
        "Strong cross-series correlation exists",
        "Yes — series causally influence each other",
        "Large (>10K timestamps per series)",
        "Moderate — GPU training 2–5 min",
        "Yes — hist/futr/static exogenous",
        "Max accuracy on multivariate benchmark",
    ],
    "Choose NHITS when...": [
        "Single series or few independent series",
        "Series are largely independent",
        "No known causal pathway",
        "Small — few thousand timesteps",
        "Low — CPU training sufficient",
        "Strong seasonality decomposition needed",
        "Interpretable hierarchical decomposition",
    ],
})

print(decision_table.to_string(index=False))

## 8. Final Summary

Record your benchmark results and compare to published numbers:

In [ ]:
print("="*65)
print("BENCHMARK RESULTS SUMMARY — ETTm1 h=96")
print("="*65)
print(f"\n{'Model':<15} {'Your MAE':>10} {'Your MSE':>10} {'Pub. MAE':>10} {'Pub. MSE':>10}")
print("-"*65)
print(f"{'NHITS':<15} {nhits_mae:>10.4f} {nhits_mse:>10.4f} {'0.3800':>10} {'0.3450':>10}")
print(f"{'XLinear':<15} {xlinear_mae:>10.4f} {xlinear_mse:>10.4f} {'0.3550':>10} {'0.3160':>10}")
print("="*65)
print()
print("Notes:")
print("  - Published numbers from datasciencewithmarco.com XLinear review")
print("  - Your numbers may differ slightly due to random seed and training steps")
print("  - XLinear should outperform NHITS by ~5-10% on ETTm1 with full training")
print()
print("The multivariate XLinear advantage is largest for OT (oil temperature),")
print("which is physically downstream of the load variables HUFL, MUFL, LUFL.")

## Next Steps

- **Exercises:** `exercises/01_xlinear_exercises.py` — verify prediction shapes, reproduce MAE comparison, ablate `hidden_size`
- **Module 6:** Production patterns — serving XLinear forecasts, retraining schedules, monitoring